# QPINN: stationary Hamilton-Jacobi

Domain: $\Omega=(-1,1)^2$ with homogeneous Dirichlet boundary condition $u=0$ on $\partial\Omega$.

PDE:

$$
-\Delta u(x)+\rho\|\nabla u(x)\|_2^\beta-q(x)=\alpha_0,
\qquad x=(x_1,x_2)\in\Omega.
$$

Source:

$$
q(x_1,x_2)=\sum_{(m,n)} c_{mn}
\cos\left(\frac{m\pi x_1}{2}\right)
\cos\left(\frac{n\pi x_2}{2}\right),
$$

with modes $(1,1),(1,3),(3,1),(1,5),(5,1)$ and coefficients defined below. This PDE has no closed-form analytical solution in the notebook; plots use a finite-difference Newton-Krylov reference solution.

Models: Ordinary QFM, Ordinary QCM, exact $B_2$-QCM, and randomized $B_2$-QCM. The shared circuit, PDE loss, training, and plotting code lives in the `qmps` package.


## Parameters

Set the PDE, circuit, training, output-directory, and figure-resolution parameters.


In [ ]:
from __future__ import annotations

import qmps as qmp
import qmps.experiments as qexp
from qmps.problems import StationaryHamiltonJacobiProblem

# Runtime.
USE_DOUBLE_PRECISION = False
SEED = 71

# Problem and circuit size.
DIM = 2
N_QUBITS = 2
N_UPLOAD_LAYERS = 2
STRONG_LAYERS_PER_BLOCK = 3

# Circuit encoding and output scaling.
EXP_BASE = 3.0
ENCODING_SCALE = 1.0
OUTPUT_SCALE = 0.75
DIFF_GENERATOR_PER_LAYER = False
ACOS_EPS = 1.0e-6

# Stationary viscous Hamilton-Jacobi PDE.
ALPHA0 = 0.0
HJ_RHO = 0.15
HJ_GRAD_POWER = 1.5
HJ_GRAD_EPS = 1.0e-12
HJ_SOURCE_MODE_COEFFICIENTS = {
    (1, 1): 0.50,
    (1, 3): 0.15,
    (3, 1): 0.15,
    (1, 5): 0.20,
    (5, 1): 0.20,
}
REFERENCE_NEWTON_TOL = 1.0e-10

# Randomized B2 average.
RANDOMIZED_B2_SAMPLES = 6

# Training.
TRAINING_STEPS = 300
N_RUNS = 10
INTERIOR_BATCH = 50
BOUNDARY_BATCH = 50
LR = 1.0e-1
LR_DECAY = 0.99
LR_MIN = 1.0e-4
INTERIOR_MARGIN = 0.96
GRAD_CLIP = 5.0
SOFT_BOUNDARY_WEIGHT = 10.0

# Evaluation and plotting.
EVAL_GRID_N = 50
FIGURE_DIR = "figures"
TRAINING_DATA_DIR = "training_data"
FIGURE_DPI = 300  # Use 600 for high-resolution export.
FIGURE_PREFIX = "qpinn_hj"

runtime = qexp.setup_runtime(
    USE_DOUBLE_PRECISION,
    SEED,
    figure_dir=FIGURE_DIR,
    data_dir=TRAINING_DATA_DIR,
)
problem = StationaryHamiltonJacobiProblem(
    alpha0=ALPHA0,
    rho=HJ_RHO,
    grad_power=HJ_GRAD_POWER,
    grad_eps=HJ_GRAD_EPS,
    reference_newton_tol=REFERENCE_NEWTON_TOL,
    source_mode_coefficients=HJ_SOURCE_MODE_COEFFICIENTS,
)
qfm_config = qmp.QFMConfig(
    dim=DIM,
    n_qubits=N_QUBITS,
    n_upload_layers=N_UPLOAD_LAYERS,
    strong_layers_per_block=STRONG_LAYERS_PER_BLOCK,
    exp_base=EXP_BASE,
    encoding_scale=ENCODING_SCALE,
    output_scale=OUTPUT_SCALE,
    diff_generator_per_layer=DIFF_GENERATOR_PER_LAYER,
    acos_eps=ACOS_EPS,
    randomized_b2_samples=RANDOMIZED_B2_SAMPLES,
)
training_config = qexp.TrainingConfig(
    steps=TRAINING_STEPS,
    n_runs=N_RUNS,
    interior_batch=INTERIOR_BATCH,
    boundary_batch=BOUNDARY_BATCH,
    lr=LR,
    lr_decay=LR_DECAY,
    lr_min=LR_MIN,
    interior_margin=INTERIOR_MARGIN,
    grad_clip=GRAD_CLIP,
)
MODEL_SPECS = {
    "$B_2$-QCM": {"group": "hyperoctahedral", "input_map": "chebyshev"},
    "Randomized $B_2$-QCM": {
        "group": "randomized_hyperoctahedral",
        "input_map": "chebyshev",
        "group_sample_count": RANDOMIZED_B2_SAMPLES,
    },
    "Ordinary QCM": {"group": "none", "input_map": "chebyshev"},
    "Ordinary QFM": {"group": "none", "input_map": "raw"},
}

for model_name, spec in MODEL_SPECS.items():
    if "QFM" in model_name:
        assert spec["input_map"] == "raw", f"{model_name} must use raw input."
    if "QCM" in model_name:
        assert spec["input_map"] == "chebyshev", f"{model_name} must use Chebyshev input."


## Models And Losses

Build the four QPINN models and define the hard/soft loss functions.


In [ ]:
def make_models(seed: int = SEED, hard_boundary: bool = True):
    return qmp.make_qfm_models(
        qfm_config,
        MODEL_SPECS,
        seed=seed,
        hard_boundary=hard_boundary,
        dtype=runtime.dtype,
        device=runtime.device,
    )


def hard_loss(model, interior_x, boundary_x):
    return problem.loss(model, interior_x, boundary_x, boundary_weight=0.0)


def soft_loss(model, interior_x, boundary_x):
    return problem.loss(model, interior_x, boundary_x, boundary_weight=SOFT_BOUNDARY_WEIGHT)


loss_functions = qexp.make_loss_functions(hard_loss, soft_loss)
models = make_models(seed=SEED, hard_boundary=True)
qexp.print_parameter_counts(models)


## Training

Train the hard-constraint and soft-constraint versions of each model.


In [ ]:
result = qexp.train_boundary_comparison(
    make_models=make_models,
    loss_functions=loss_functions,
    model_names=list(MODEL_SPECS),
    config=training_config,
    runtime=runtime,
    dim=DIM,
    seed=SEED,
)
qexp.attach_final_errors(result, problem, runtime, dim=DIM, grid_n=EVAL_GRID_N)
training_data_path = qexp.save_training_data(result, runtime.data_dir, FIGURE_PREFIX)
print(f"saved training data: {training_data_path}")


## Figures

Save the training-loss and solution-comparison figures.


In [ ]:
figure_paths = [
    qexp.plot_training_losses(
        result,
        runtime.figure_dir,
        FIGURE_PREFIX,
        dpi=FIGURE_DPI,
    ),
    qexp.plot_solution_panels(
        result,
        problem,
        runtime,
        dim=DIM,
        grid_n=EVAL_GRID_N,
        figure_prefix=FIGURE_PREFIX,
        dpi=FIGURE_DPI,
    ),
]
for figure_path in figure_paths:
    print(f"saved figure: {figure_path}")
